In [2]:
# Instalar psycopg2-binary directamente
!pip install psycopg2-binary

  Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl (2.8 MB)


In [3]:
import psycopg2

try:
    # 1. Establecer conexión
    conexion = psycopg2.connect(
        database="gestion_info",
        user="postgres",
        password="root",  # <-- Escriba su contraseña aquí
        host="localhost",
        port=5432
    )
    print("¡Conexión establecida con éxito!")
    
    # 2. Crear cursor y ejecutar consulta
    cursor = conexion.cursor()
    cursor.execute("SELECT version();")
    
    # 3. Mostrar versión del motor
    version_bd = cursor.fetchone()
    print(f"Versión de PostgreSQL detectada: {version_bd[0]}")
    
    # 4. Cerrar recursos
    cursor.close()
    conexion.close()
    print("Conexión cerrada correctamente.")
    
except Exception as e:
    print(f"Fallo al conectar a la base de datos: {e}")

¡Conexión establecida con éxito!
Versión de PostgreSQL detectada: PostgreSQL 16.3, compiled by Visual C++ build 1938, 64-bit
Conexión cerrada correctamente.


In [5]:
import psycopg2
from psycopg2 import pool

DB_CONFIG = {
    "database": "gestion_info",
    "user": "postgres",
    "password": "root",  # <-- Escriba su contraseña aquí
    "host": "localhost",
    "port": 5432
}

try:
    # 1. Instanciar pool de conexiones
    pool_conexiones = psycopg2.pool.SimpleConnectionPool(1, 5, **DB_CONFIG)
    print("Pool de conexiones creado con éxito.\n")
    
    # 2. Obtener conexión limpia y cursor
    conn = pool_conexiones.getconn()
    cursor = conn.cursor()
    
    # Limpiar registros previos
    cursor.execute("TRUNCATE TABLE estudiantes;")
    conn.commit()
    
    # ================= INICIO DE LA TRANSACCIÓN =================
    cursor.execute("BEGIN;")
    print("Transacción principal iniciada...")
    
    # --- OPERACIÓN 1: Registrar al Estudiante A (Exitoso) ---
    cursor.execute("INSERT INTO estudiantes (id, nombre) VALUES (1, 'Ana Pérez');")
    print("-> Estudiante 1 (Ana) registrado correctamente.")
    
    # --- ESTABLECER SAVEPOINT ---
    cursor.execute("SAVEPOINT sp_estudiante_b;")
    print("-> SAVEPOINT 'sp_estudiante_b' creado.")
    
    try:
        # --- OPERACIÓN 2: Registrar al Estudiante B forzando un ID duplicado (Falla) ---
        cursor.execute("INSERT INTO estudiantes (id, nombre) VALUES (1, 'Carlos Gómez');")
        print("-> Estudiante 2 registrado.")
        cursor.execute("RELEASE SAVEPOINT sp_estudiante_b;")
        
    except Exception as e_interno:
        # En caso de error interno, revertimos únicamente la operación 2
        print(f"\n[!] Error detectado en la Operación 2: {e_interno}")
        cursor.execute("ROLLBACK TO SAVEPOINT sp_estudiante_b;")
        print("-> Rollback parcial aplicado al SAVEPOINT. Operación 1 queda a salvo.")
    
    # --- CONFIRMAR TRANSACCIÓN ---
    cursor.execute("COMMIT;")
    print("\nTransacción confirmada globalmente (COMMIT).")
    
    # 3. Comprobar registros guardados
    cursor.execute("SELECT * FROM estudiantes;")
    resultado = cursor.fetchall()
    print(f"\nRegistros en la tabla 'estudiantes': {resultado}")
    
    # 4. Devolver conexión al pool y liberar
    cursor.close()
    pool_conexiones.putconn(conn)
    print("\nConexión retornada al pool correctamente.")
    
    pool_conexiones.closeall()
    print("¡Éxitos! Estás listo para el examen.")
    
except Exception as e_global:
    print(f"Error general en el proceso: {e_global}")

Pool de conexiones creado con éxito.

Transacción principal iniciada...
-> Estudiante 1 (Ana) registrado correctamente.
-> SAVEPOINT 'sp_estudiante_b' creado.

[!] Error detectado en la Operación 2: duplicate key value violates unique constraint "estudiantes_pkey"
DETAIL:  Key (id)=(1) already exists.

-> Rollback parcial aplicado al SAVEPOINT. Operación 1 queda a salvo.

Transacción confirmada globalmente (COMMIT).

Registros en la tabla 'estudiantes': [(1, 'Ana Pérez')]

Conexión retornada al pool correctamente.
¡Éxitos! Estás listo para el examen.
